# TKO_7092 Evaluation of Machine Learning Methods 2026

## Exercise 3

## IMPORTANT

This exercise involves using AI. Use only the *Study Chat* service provided by the University of Turku at [https://ai.utu.fi/en](https://ai.utu.fi/en). **Do not use any other AI service!** Before starting, remember to carefully read the guidelines for using AI ([https://intranet.utu.fi/en/sites/ai/guidelines/Pages/default.aspx](https://intranet.utu.fi/en/sites/ai/guidelines/Pages/default.aspx)) and the terms of use. **Do not share any personal information or copyrighted material with AI.**

Save all your discussions (including your prompts and AI's output) as well as the name of the model you used.

## Instructions

The deadline of this exercise is **Wednesday 18 February 2026 at 11:59 PM**. Please contact Juho Heimonen (juaheim@utu.fi) if you have any questions about the exercise. Remember to follow all the general exercise guidelines that are stated in Moodle.

The exercise has several parts, all of which concern the letter below. You will take the role of a data scientist who has been assigned to solve the problem described in the letter. You have an AI tool to assist you, but you alone are responsible for the quality of the solution.

#### 1

Ask AI to write code to solve the task. Analyse which parts of the AI-generated code are correct and which are incorrect. Pay particular attention to the key parts of the cross-validation. You may ask AI to improve the code as many times as you like, as long as you keep analysing its output.

#### 2

Implement the required leave-one-out cross-validations and run your code to get the estimates you were asked to obtain. Here it is okay to use any amount of the AI-generated code you produced above. You can use a complete, fully correct AI-written solution, you can write the implementation from scratch by yourself, or you can take some AI-generated code and complete the implementation manually.

#### 3

Write a report in which you discuss the following:

- Why did the cross-validation described in the letter fail? What is the correct way to do cross-validation here?

- Which parts of the task was AI able to code correctly and which not? Focus particularly on the core of the cross-validation.

- Which parts of the AI-generated code did you use in your implementation? Why? Explain why the selected pieces of code work correctly in your implementation.

- What results did you get with your implementation? Report the estimates and interpret the results in terms of how well the model will work in the situations described in the letter. Explain in detail why your cross-validation is the correct way to estimate the generalisation performance.

Write the report in your own words and explain everything clearly, precisely, and comprehensively. **You are not allowed to use AI to write the report for you** because this is where you show that you have understood the theory and are able to apply it. If you use AI as a teacher (i.e. to explain things to you for learning purposes), you must attach the discussions and clearly state what and how you learnt from the AI. **If there is uncertainty about how the text was produced, you may be required to explain the content of your report in a face-to-face meeting.**

#### 4

Submit the following documents to Moodle:

- The discussions with AI (including your prompts and AI's output), as PDF.

- The implementation of your cross-validation, as PDF and as a Jupyter notebook.

- The report, as PDF. (It is okay to integrate the report to the Jupyter notebook.)

Note that it is not enough to just implement the cross-validation correctly to pass this exercise. You must also explain in plain words what you have done and demonstrate that you understand how cross-validation should be performed on pair-input data. Small errors are acceptable, but you will fail this exercise if there are significant error(s) or omission(s) in the report or in the implementation.

## Letter from your client

Dear Data Scientist,

I have a long-term research project regarding a specific set of proteins. I am attempting to discover small organic compounds that can bind strongly to these proteins and thus act as drugs. I have already made laboratory experiments to measure the affinities between some proteins and drug molecules.

My colleague is working on another set of proteins, and the objectives of his project are similar to mine. He has recently discovered thousands of new potential drug molecules. He asked me if I could find the pairs that have the strongest affinities among his proteins and drug molecules. Obviously I do not have the resources to measure all the possible pairs in my laboratory, so I need to prioritise. I decided to do this with the help of machine learning, but I have encountered a problem.

Here is what I have done so far: First I trained a K-nearest neighbours regressor with the parameter value K=10 using all the 400 measurements I had already made in the laboratory with my proteins and drug molecules. They comprise of 77 target proteins and 59 drug molecules. To estimate the generalisation performance of the model, I then performed a leave-one-out cross-validation. I used C-index and got a stellar score above 90%. Finally I used the model to predict the affinities of my colleague's proteins and drug molecules. The problem is that when I selected the highest predicted affinities and my colleague tried to verify them in the lab, we found that many of them are much lower in reality. My model clearly does not work despite the high cross-validation score. We also tested the model with my proteins against my colleague's drugs (which is another task I would like to use my model for), but the model did not work there either.

Please explain why my estimation failed and how leave-one-out cross-validation should be performed to get reliable estimates. Also, please implement the leave-one-out cross-validation correctly and report the numbers I need. I want to know whether it would be a waste of my colleague's and my resources if we were to use my model any further.

The data I used to create my model is available in the files `input.data`, `output.data` and `pairs.data` for you to use. The first file contains the features of the pairs, whereas the second contains their affinities. The third file contains the identifiers of the drug and target molecules of which the pairs are composed. The files are paired, i.e. the i<sup>*th*</sup> row in each file is about the same pair.

Looking forward to hearing from you soon.

Yours sincerely, \
Bio Scientist


### my key bard is proken and the letter "o" doesn't work properly sorry for that and if some o letters are missing from my explanations.

The client's solution failed because she treated the pairs as independent samples but they are not independent and bacause of that the same protein appears in many pairs aswell as the same drug appears in many pairs. And when they removed one pair in leave one out cross validation the same protein and drug still appear in the training set. this causes data leakage and the c score to be too optimistic and then it does not measure generalization to new drugs or new proteins.

so a better way would be leave one drug out that removes all pairs containing one drug from training and does prediction only for pairs containing that drug which measures generalization to new drugs. Or leave one protein out that removes all pairs containing one protein from training and does the prediction only for pairs that doesnt contain that drug this way it meausres generalization to new proteins. 

In [6]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from lifelines.utils import concordance_index


In [7]:
# Load the data
features = np.loadtxt("input.data")        # shape (400, n_features)
affinities = np.loadtxt("output.data")     # shape (400,)
pairs = pd.read_csv("pairs.data", sep=" ", header=None, names=["drug","protein"], quotechar='"')
proteins = pairs["protein"].values
drugs = pairs["drug"].values

In [8]:
def calculate_cindex(true, pred):
    """Calculate Concordance Index (C-index)"""
    n = len(true)
    concordant = 0
    discordant = 0
    tied = 0

    for i in range(n):
        for j in range(i+1, n):
            if true[i] > true[j]:
                if pred[i] > pred[j]:
                    concordant += 1
                elif pred[i] < pred[j]:
                    discordant += 1
                else:
                    tied += 1
            elif true[i] < true[j]:
                if pred[i] < pred[j]:
                    concordant += 1
                elif pred[i] > pred[j]:
                    discordant += 1
                else:
                    tied += 1
            else:
                if pred[i] == pred[j]:
                    tied += 1

    if concordant + discordant == 0:
        return 0.5  # All pairs are tied

    return (concordant + 0.5 * tied) / (concordant + discordant + tied)


In [9]:
def type_d_loocv(features, affinities, protein_ids, drug_ids, n_neighbours=10):
    scaler= StandardScaler()
    unique_proteins= np.unique(protein_ids)
    unique_drugs= np.unique(drug_ids)

    n_samples= len(affinities)
    predictions= np.zeros(n_samples)
    mse_scores= []

    for protein in unique_proteins:
        for drug in unique_drugs:
            
            test_mask= (protein_ids == protein) & (drug_ids == drug)
            if not np.any(test_mask):
                continue
            #remove all pairs containing that protein or drug
            train_mask= (protein_ids != protein) & (drug_ids != drug)
            train_indices= np.where(train_mask)[0]
            test_indices= np.where(test_mask)[0]

            if len(train_indices)==0:
                continue
            X_train= features[train_indices]
            y_train= affinities[train_indices]

            X_test= features[test_indices]
            y_test= affinities[test_indices]

            X_train_scaled= scaler.fit_transform(X_train)
            X_test_scaled= scaler.transform(X_test)

            knn= KNeighborsRegressor(n_neighbors=min(n_neighbours, len(train_indices)))
            knn.fit(X_train_scaled,y_train.ravel())
            y_pred= knn.predict(X_test_scaled)

            predictions[test_indices] = y_pred

            mse = mean_squared_error(y_test, y_pred)
            mse_scores.append(mse)

    avg_mse = np.mean(mse_scores)
    rmse = np.sqrt(avg_mse)

    c_index = calculate_cindex(affinities, predictions)
    cindex = concordance_index(affinities, predictions)

    return {
        "mse": avg_mse,
        "rmse": rmse,
        "cindex": cindex,
        "predictions": predictions,
        'c_index': c_index
    }
            

In [10]:
def leave_one_drug_out_cv(features, affinities, drug_ids, n_neighbors=10):
    """
    Perform Leave-One-Drug-Out Cross-Validation for drug-protein affinity prediction.

    Args:
        features (np.ndarray): Feature matrix (n_samples × n_features)
        affinities (np.ndarray): Target values (n_samples × 1)
        protein_ids (np.ndarray): Array of protein identifiers (length n_samples)
        drug_ids (np.ndarray): Array of drug identifiers (length n_samples)
        n_neighbors (int): Number of neighbors for KNN (default: 10)

    Returns:
        dict: Dictionary containing:
            - 'mse': Mean Squared Error
            - 'rmse': Root Mean Squared Error
            - 'cindex': Concordance Index
            - 'predictions': Array of predicted values
    """
    # Initialize model and scaler
    scaler = StandardScaler()

    # Get unique drugs
    unique_drugs = np.unique(drug_ids)
    n_samples = len(affinities)
    predictions = np.zeros(n_samples)
    mse_scores = []

    # Perform LODO-CV
    for drug in unique_drugs:
        # Get indices of all pairs with this drug
        test_mask = (drug_ids == drug)
        test_indices = np.where(test_mask)[0]
        train_indices = np.where(~test_mask)[0]

        # Split data
        X_train = features[train_indices]
        y_train = affinities[train_indices]
        X_test = features[test_indices]
        y_test = affinities[test_indices]

        # Standardize features (fit on training only)
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Train and predict
        knn = KNeighborsRegressor(n_neighbors=min(n_neighbors, X_train_scaled.shape[0]))
        knn.fit(X_train_scaled, y_train.ravel())
        y_pred = knn.predict(X_test_scaled)

        # Store predictions and calculate MSE
        predictions[test_indices] = y_pred
        mse = mean_squared_error(y_test, y_pred)
        mse_scores.append(mse)

    # Calculate overall performance metrics
    avg_mse = np.mean(mse_scores)
    rmse = np.sqrt(avg_mse)
    cindex = calculate_cindex(affinities, predictions)
    c_index = concordance_index(affinities, predictions)

    return {
        'mse': avg_mse,
        'rmse': rmse,
        'cindex': cindex,
        'predictions': predictions,
        'c_index': c_index
    }


In [ ]:
# Run D tupe loocv
results = type_d_loocv(features, affinities,protein_ids=proteins,drug_ids=drugs)

# Print results
print(f"Results for unseen drugs and prteins pairs:")
print(f"Mean Squared Error: {results['mse']:.4f}")
print(f"Root Mean Squared Error: {results['rmse']:.4f}")
print(f"Concordance Index: {results['cindex']:.4f}")
print(f"C-index: {results['c_index']:.4f}")

results for unseen drugs and prteins pairs:
Mean Squared Error: 0.0752
Root Mean Squared Error: 0.2742
Concordance Index: 0.5130
C-index: 0.5130


In [14]:
# Run LOPO-CV
result = leave_one_drug_out_cv(features, affinities, proteins,n_neighbors=10)

# Print results
print(f"Results for known proteins - unseen drugs:")
print(f"Mean Squared Error: {result['mse']:.4f}")
print(f"Root Mean Squared Error: {result['rmse']:.4f}")
print(f"Concordance Index: {result['cindex']:.4f}")
print(f"C-index: {result['c_index']:.4f}")


Results for known proteins - unseen drugs:
Mean Squared Error: 0.0154
Root Mean Squared Error: 0.1240
Concordance Index: 0.8287
C-index: 0.8287


## Why
### scientist cross validation
 The bio scientist crooss validation produced an overly optimistic estimate of the models performance because it didn't properly simulate the real prediction scenario. The leave one out cross validation was performed at the level of individual drug prrotein pairs which means that when one pair was left out for testing the training data still contained other pairs with the same drug or the same protein. And as a result the model had already seen information about both the drug and the protein in training which causes information leakage between the training and testing sets and allows the model to indirectly memorize similarities between drugs and proteins rather than truly generalizing to new ones. And because of this the cross validation produced a very high C-index even though the model doesn't generalize well to a new data. 
### D type 
The bio scientist wants to predict affinities between new drug molecules and new proteins discovered by a colleague. From the perspective of the trained model both the drugs and proteins are unseen duringt training. Type D cross validation corresponds to this scenario where the evaluatioon method must stimulate a situatin where the mdel predicts interactins fr drugs and proteins that are nt presents in the training data. In d type approach when spesific drug protein pair is used as the test instance all pairs containing that drug and all pairs containing that protein are remved from the training data, which ensures that the mdel cannt rely on previusly observed infrmatin about the same drug r prteins.

# What I changed
## from my last solution
In my first solution my cross validation didn't correctly remove all relevant information from the training set. The test selection was done incorrectly and didn't isolate a single drug protein pair. And because of that the training data still contained pairs that shared either the same drug or same protein as the test pair, which allowed the model to access information about the test entities in training which lead to information leakage and overly optimistic performance estimates. 
So I modified the cross validitation to be Tupe D (nested) leave one out cross validation. Where the test set contains exactly one drug protein pair which is defined by selecting rows where both the drug and protein identifier match the chosen pair. Then the training set is done by removing all rows that containing either the same drug or protein, which guarantees that the model doesn't see the drug or the protein during training. After training the Knn with K=10 on the remaining data, prediction is done for the test pair. This process is repeated for every pair in the dataset and predictions are pooled to compute the final c-index. I decided to keep the leave one drug out cross validation for the scientist secndary task: We also tested the model with my proteins against my colleague's drugs (which is another task I would like to use my model for), but the model did not work there either.

### intreperting c-index scres and results
The c-index for unseen drugs and proteins (D type crss validation) is 0.51 which is almst random. This means the mdel can't crrectly rankaffinities when both drugs and proteins are new meaning the mdel cannot generalise well to collagues drugs and proteins. this explains why the labratory expirements failed to cnfirm the high predicted affinities despite the initially high cross-validation score.
the c-index fr known prteind unseen drugs is 0.83 (good ranking performance) which means the model ranks affinities crrectly most of the time and the model generalises resnably well to new drugs if proteins are knwn

overall these results suggest that the mdel shuld nt be used to predict affinities between the cllagues drugs and proteins because the performance in the unseen drug-unseen prtein scenari is close to randm. But the mdel might still be useful for predicting affinities between the scientist prteins and new drug molecules.